In [2]:
import os

import cenpy
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


/opt/anaconda3/envs/dindex/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
acs_api = cenpy.remote.APIConnection("ACSDT5Y2021")

acs_variables = [
    "B15003_001E",
    "B15003_017E", "B15003_018E", "B15003_019E", "B15003_020E",
    "B15003_021E", "B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E",
    "B17001_001E", "B17001_002E",
    "B23025_002E", "B23025_005E",
    "B19058_001E", "B19058_002E",
    "B19056_001E", "B19056_002E",
    "B19001_001E",
    "B19001_002E", "B19001_003E", "B19001_004E", "B19001_005E", "B19001_006E",
]

request_args = {
    "cols": ["NAME", "GEO_ID", *acs_variables],
    "geo_unit": "county",
    "geo_filter": {"state": "*"},
}

census_key = "918876e93cf30566c02b367fcad29644d861817e"
if census_key:
    request_args["apikey"] = census_key

county_stats = acs_api.query(**request_args)
county_stats[acs_variables] = county_stats[acs_variables].apply(pd.to_numeric, errors="coerce")
county_stats["GEOID"] = (
    county_stats["state"].astype(str).str.zfill(2)
    + county_stats["county"].astype(str).str.zfill(3)
)

county_stats.shape


(3221, 29)

In [4]:
some_college_or_more_fields = [
    "B15003_019E",
    "B15003_020E",
    "B15003_021E",
    "B15003_022E",
    "B15003_023E",
    "B15003_024E",
    "B15003_025E",
]

under_30k_fields = [
    "B19001_002E",
    "B19001_003E",
    "B19001_004E",
    "B19001_005E",
    "B19001_006E",
]

def ratio(numerator, denominator):
    denominator = denominator.mask(denominator <= 0)
    return numerator.div(denominator)

some_college_or_more = county_stats[some_college_or_more_fields].sum(axis=1)
low_income_households = county_stats[under_30k_fields].sum(axis=1)

county_stats["low_education_rate"] = 1 - ratio(
    some_college_or_more,
    county_stats["B15003_001E"],
)
county_stats["poverty_rate"] = ratio(
    county_stats["B17001_002E"],
    county_stats["B17001_001E"],
)
county_stats["unemployment_rate"] = ratio(
    county_stats["B23025_005E"],
    county_stats["B23025_002E"],
)
county_stats["low_income_hh_rate"] = ratio(
    low_income_households,
    county_stats["B19001_001E"],
)
county_stats["ssi_hh_rate"] = ratio(
    county_stats["B19056_002E"],
    county_stats["B19056_001E"],
)
county_stats["snap_pa_hh_rate"] = ratio(
    county_stats["B19058_002E"],
    county_stats["B19058_001E"],
)


In [5]:
indicator_columns = [
    "poverty_rate",
    "unemployment_rate",
    "low_education_rate",
    "low_income_hh_rate",
    "ssi_hh_rate",
    "snap_pa_hh_rate",
]

county_stats[["NAME", "GEOID", *indicator_columns]].head()


,NAME,GEOID,poverty_rate,unemployment_rate,low_education_rate,low_income_hh_rate,ssi_hh_rate,snap_pa_hh_rate
0,"Autauga County, Alabama",01001,0.135785,0.027296,0.431741,0.260706,0.060624,0.107980
1,"Baldwin County, Alabama",01003,0.092049,0.036685,0.363613,0.206147,0.045957,0.071281
2,"Barbour County, Alabama",01005,0.264719,0.086242,0.610058,0.438050,0.105964,0.259793
3,"Bibb County, Alabama",01007,0.169429,0.097068,0.633805,0.305520,0.067768,0.179585
4,"Blount County, Alabama",01009,0.132366,0.060130,0.514754,0.275023,0.087418,0.109061


In [6]:
normalized_columns = [f"{name}_norm" for name in indicator_columns]

for column in indicator_columns:
    values = county_stats[column].astype(float)
    lower = values.min(skipna=True)
    upper = values.max(skipna=True)
    county_stats[f"{column}_norm"] = (values - lower) / (upper - lower)

county_stats["economic_distress_index"] = county_stats[
    normalized_columns
].mean(axis=1)

county_stats[
    indicator_columns + normalized_columns + ["economic_distress_index"]
].describe().loc[["min", "max", "mean"]]


,poverty_rate,unemployment_rate,low_education_rate,low_income_hh_rate,ssi_hh_rate,snap_pa_hh_rate,poverty_rate_norm,unemployment_rate_norm,low_education_rate_norm,low_income_hh_rate_norm,ssi_hh_rate_norm,snap_pa_hh_rate_norm,economic_distress_index
min,0.012048,0.000000,0.092316,0.030303,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.047485
max,0.670737,0.340806,0.961165,0.832910,0.352227,0.630624,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,0.767763
mean,0.151739,0.054377,0.462111,0.273047,0.057440,0.139425,0.212074,0.159555,0.425615,0.302444,0.163077,0.22109,0.247309


In [7]:
index_table = county_stats[
    [
        "GEOID",
        "NAME",
        "state",
        "county",
        *indicator_columns,
        *normalized_columns,
        "economic_distress_index",
    ]
].copy()

index_table["GEOID"] = index_table["GEOID"].astype(str).str.zfill(5)

index_table.head()


,GEOID,NAME,state,county,poverty_rate,unemployment_rate,low_education_rate,low_income_hh_rate,ssi_hh_rate,snap_pa_hh_rate,poverty_rate_norm,unemployment_rate_norm,low_education_rate_norm,low_income_hh_rate_norm,ssi_hh_rate_norm,snap_pa_hh_rate_norm,economic_distress_index
0,01001,"Autauga County, Alabama",01,001,0.135785,0.027296,0.431741,0.260706,0.060624,0.107980,0.187853,0.080092,0.390661,0.287069,0.172117,0.171226,0.214836
1,01003,"Baldwin County, Alabama",01,003,0.092049,0.036685,0.363613,0.206147,0.045957,0.071281,0.121455,0.107642,0.312249,0.219091,0.130476,0.113033,0.167324
2,01005,"Barbour County, Alabama",01,005,0.264719,0.086242,0.610058,0.438050,0.105964,0.259793,0.383597,0.253053,0.595895,0.508028,0.300840,0.411962,0.408896
3,01007,"Bibb County, Alabama",01,007,0.169429,0.097068,0.633805,0.305520,0.067768,0.179585,0.238930,0.284819,0.623225,0.342904,0.192399,0.284773,0.327842
4,01009,"Blount County, Alabama",01,009,0.132366,0.060130,0.514754,0.275023,0.087418,0.109061,0.182663,0.176435,0.486204,0.304907,0.248186,0.172941,0.261889


In [8]:
boundary_url = (
    "https://www2.census.gov/geo/tiger/GENZ2024/shp/"
    "cb_2024_us_county_500k.zip"
)

county_shapes = gpd.read_file(boundary_url)
excluded_state_fips = {"02", "15", "60", "66", "69", "72", "78"}

county_shapes["STATEFP"] = county_shapes["STATEFP"].astype(str).str.zfill(2)
county_shapes["GEOID"] = county_shapes["GEOID"].astype(str).str.zfill(5)
county_shapes = county_shapes[
    ~county_shapes["STATEFP"].isin(excluded_state_fips)
].copy()

county_map = county_shapes.merge(
    index_table,
    on="GEOID",
    how="left",
    suffixes=("", "_acs"),
).to_crs(4326)

len(county_map), county_map["economic_distress_index"].notna().sum()


(3109, np.int64(3100))

In [9]:
county_map.loc[
    county_map["economic_distress_index"].isna(),
    ["GEOID", "NAME", "STATE_NAME"],
]


,GEOID,NAME,STATE_NAME
17,09190,Western Connecticut,Connecticut
284,09170,South Central Connecticut,Connecticut
285,09180,Southeastern Connecticut,Connecticut
762,09110,Capitol,Connecticut
902,09140,Naugatuck Valley,Connecticut
1375,09130,Lower Connecticut River Valley,Connecticut
1462,09160,Northwest Hills,Connecticut
2149,09120,Greater Bridgeport,Connecticut
2798,09150,Northeastern Connecticut,Connecticut


In [10]:
import matplotlib as mpl
import mapclassify
from matplotlib.colors import ListedColormap
from cmcrameri import cm

gdf = county_map.to_crs(5070).copy()
value_col = "economic_distress_index"
valid_mask = gdf[value_col].notna()
valid_values = gdf.loc[valid_mask, value_col].astype(float)

class_count = min(5, int(valid_values.nunique()))

if class_count >= 2:
    classifier = mapclassify.NaturalBreaks(valid_values.to_numpy(), k=class_count)
    gdf["_class"] = np.nan
    gdf.loc[valid_mask, "_class"] = classifier.yb.astype(float)
    bins = np.asarray(classifier.bins, dtype=float)
else:
    classifier = None
    class_count = 1
    gdf["_class"] = np.nan
    gdf.loc[valid_mask, "_class"] = 0
    bins = np.array([float(valid_values.max())])

colors = cm.lajolla_r(np.linspace(0.18, 0.92, class_count))
cmap = ListedColormap(colors)

state_geometry = gdf[["STATEFP", "geometry"]].dropna(subset=["STATEFP"]).copy()
states = state_geometry.dissolve(by="STATEFP")

background = "#F7F6F1"
fig = plt.figure(figsize=(16, 10), facecolor=background)
ax = fig.add_axes([0.015, 0.16, 0.97, 0.72])
ax.set_facecolor(background)

gdf.plot(
    ax=ax,
    column="_class",
    cmap=cmap,
    vmin=-0.5,
    vmax=class_count - 0.5,
    edgecolor="#FFFFFF",
    linewidth=0.18,
    missing_kwds={
        "color": "#D9D9D6",
        "edgecolor": "#FFFFFF",
        "linewidth": 0.15,
    },
)

states.boundary.plot(
    ax=ax,
    color="#4A4A4A",
    linewidth=0.65,
    alpha=0.85,
    zorder=3,
)

ax.set_title(
    "Economic Distress Index by U.S. County",
    fontsize=22,
    fontweight="bold",
    loc="left",
    pad=16,
    color="#202020",
)
ax.set_axis_off()

minimum = float(valid_values.min())
maximum = float(valid_values.max())

if classifier is not None:
    lower_bounds = np.r_[minimum, bins[:-1]]
    labels = [f"{low:.2f}–{high:.2f}" for low, high in zip(lower_bounds, bins)]
else:
    labels = [f"{minimum:.2f}–{maximum:.2f}"]

legend_ax = fig.add_axes([0.25, 0.085, 0.50, 0.028])
norm = mpl.colors.BoundaryNorm(
    np.arange(-0.5, class_count + 0.5, 1),
    cmap.N,
)
mappable = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable.set_array([])

colorbar = fig.colorbar(
    mappable,
    cax=legend_ax,
    orientation="horizontal",
    ticks=np.arange(class_count),
)

colorbar.ax.set_xticklabels(labels, fontsize=9)
colorbar.outline.set_visible(False)
colorbar.ax.tick_params(length=0, pad=7, colors="#444444")
colorbar.ax.set_title(
    "Economic Distress Index — Natural Breaks",
    fontsize=10.5,
    fontweight="semibold",
    loc="left",
    pad=8,
    color="#333333",
)

fig.text(0.25, 0.055, "Lower distress", fontsize=9, color="#777777", ha="left")
fig.text(0.75, 0.055, "Higher distress", fontsize=9, color="#777777", ha="right")
fig.text(
    0.78,
    0.095,
    "Gray indicates no available index value",
    fontsize=9,
    color="#777777",
    ha="left",
)
fig.text(
    0.015,
    0.018,
    "Source: 2017–2021 ACS 5-Year county estimates",
    fontsize=8.5,
    color="#777777",
    ha="left",
)

plt.show()


findfont: Failed to find font weight semibold, now using 700.
